# Time-travel debug — rewind, fork, compare

The checkpointer keeps a snapshot after every node. That gives you two superpowers:
* **Replay** — re-run from a specific checkpoint to reproduce a bug deterministically.
* **Fork** — copy the timeline onto a new `thread_id`, patch the past, and run an alternate future — without touching the original.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / '04-human-in-the-loop'))
os.environ.setdefault('LLM_CACHE_ONLY', '1')

from hitl import build_research_graph, Checkpointer, Command, get_question

graph = build_research_graph(); cp = Checkpointer()
state = graph.run({'question': get_question('q01'), 'question_id': 'q01'},
                  thread_id='orig', checkpointer=cp)
state = graph.resume(thread_id='orig',
                     command=Command(resume={'approved': True, 'reviewer': 'first-take'}),
                     checkpointer=cp)
[(c.step, c.node, c.state.get('draft', '')[:40]) for c in cp.history('orig')]

In [ ]:
# Fork at the post-interrupt checkpoint and resume with the OPPOSITE decision.
interrupt_cp = next(c for c in reversed(cp.history('orig')) if c.interrupt is not None)
cp.fork('orig', from_step=interrupt_cp.step, new_thread_id='fork-deny')
alt = graph.resume(thread_id='fork-deny',
                   command=Command(resume={'approved': False, 'reviewer': 'denied-take'}),
                   checkpointer=cp)

print('orig published:', state['final_answer'])
print('fork published:', alt['final_answer'])
print('originals are still', [c.state.get('draft','')[:40] for c in cp.history('orig')])

## Why this is the killer debugging feature

In production: a user reports a bad agent answer. You pull the failed thread from your `PostgresSaver`, replay it locally to confirm, fork at the suspect node, try fixes — all without spending another dollar on the deterministic upstream steps. The fork is then your regression test.